In [1]:
import os
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler
from utils import *

os.makedirs('./results/many_well_2', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

TORCH DEVICE: cuda


In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
class MLPScore(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(2, 256),
            nn.Tanh(),
            nn.Linear(256,256),
            nn.Tanh(),
            nn.Linear(256,256),
            nn.Tanh(),
            nn.Linear(256,2)
        )

    def forward(self, x):
        score = self.main(x)

        return score

In [5]:
def jacobian(f, x):
    """
    Computes the Jacobian
    """
    B, N = x.shape
    y = f(x)
    jacobian = list()
    for i in range(N):
        v = torch.zeros_like(y)
        v[:, i] = 1.
        dy_i_dx = autograd.grad(y, x, grad_outputs=v, retain_graph=True,
                                create_graph=True, allow_unused=True)[0]  # shape [B, N]
        jacobian.append(dy_i_dx)
    jacobian = torch.stack(jacobian, dim=2).requires_grad_()
    return jacobian

In [6]:
def score_matching(model, samples):
    samples.requires_grad_(True)
    logp = model(samples)
    norm_jacobian = torch.norm(logp, dim=-1) ** 2 / 2.
    jacob_mat = jacobian(model, samples)
    #print(jacob_mat.shape)
    trace_jacobian = torch.diagonal(jacob_mat, dim1=-2, dim2=-1).sum(-1)
    loss = (trace_jacobian + norm_jacobian).mean(-1)
    return loss

In [7]:
dim = 2
filename = "many_well_dataset_2_110000_samples.pkl"
f = open(filename, 'rb')
dataset = pickle.load(f,encoding="latin1")
# test_filename = "many_well_dataset_16_20000_test_samples.pkl"
# t = open(test_filename,'rb')
# testset = pickle.load(t,encoding="latin1")

In [8]:
dataset.shape

torch.Size([110000, 2])

In [9]:
model = MLPScore().to(torch_device)
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)


In [10]:
PATH = "./results/many_well_2/score_estimation_model_100000.pt"

In [11]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [12]:
def grad_dist(x):
    x = x.cpu().detach()
    score = []
    for i in range(x.size()[1]//2):
        y1 = - 4*x[:,2*i]**3 + 12*x[:,2*i] + 0.5
        y2 = - x[:,2*i+1]
        score.append(y1)
        score.append(y2)
    return torch.tensor(np.array(score).T)

In [16]:
neg_grad = -grad_dist(dataset)

In [17]:
neg_grad.shape

torch.Size([110000, 2])

In [18]:
array = [dataset.cpu().detach().numpy(), neg_grad.numpy()]

In [ ]:
# neg_grad_model = - model(dataset).cpu().detach().numpy()

In [ ]:
# neg_grad_model.shape

In [ ]:
# array = [dataset.cpu().detach().numpy(), neg_grad_model]

In [19]:
output = open("many_well_data_2_dim_with_neg_score_through_log_pdf_110000_samples.pkl", 'wb')
pickle.dump(array, output)
output.close()

In [ ]:
#many_well_data_2 dim_with_neg_score_through_log_pdf.pkl
#many_well_data_32 dim_with_neg_score_through_score_model_10000.pkl
# many_well_data_16_dim_with_neg_score_through_score_model_100000_120000_samples.pkl